In [ ]:
import csv
import hashlib
import math
import os
from pathlib import Path

import torch
from PIL import Image
from torchvision.transforms.functional import pil_to_tensor
from torchvision.models.detection import (
    FasterRCNN_MobileNet_V3_Large_FPN_Weights,
    fasterrcnn_mobilenet_v3_large_fpn,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor


def create_model(config, pretrained=True):
    weights = FasterRCNN_MobileNet_V3_Large_FPN_Weights.COCO_V1 if pretrained else None
    model = fasterrcnn_mobilenet_v3_large_fpn(
        weights=weights, weights_backbone=None, min_size=config["image_size"],
        max_size=config["image_size"], box_score_thresh=config["confidence"],
        box_nms_thresh=config["nms"], box_detections_per_img=300,
    )
    features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(features, 2)
    return model

CONFIG = {'image_size': 640, 'epochs': 3, 'batch_size': 4, 'learning_rate': 0.0001, 'horizontal_flip': 0.5, 'confidence': 0.3, 'nms': 0.5}
EXPECTED_WEIGHTS = '0f3bba3d885106ccd5cfae0ab0f242895dac4eb68754897da71f2479d987a1a4'

def predict(data_root, weight_path, out):
    with weight_path.open("rb") as handle:
        if hashlib.file_digest(handle, "sha256").hexdigest() != EXPECTED_WEIGHTS:
            raise ValueError("Checkpoint does not match the frozen baseline")
    device = torch.device("cpu")
    torch.set_num_threads(2)
    model = create_model(CONFIG, pretrained=False)
    model.load_state_dict(torch.load(weight_path, map_location=device, weights_only=True))
    model.eval()
    paths = sorted((data_root / "test").glob("*.jpg"))
    if not paths:
        raise FileNotFoundError("No Kaggle test images")
    with (data_root / "sample_submission.csv").open(newline="") as handle:
        sample_ids = [row["image_id"] for row in csv.DictReader(handle)]
    images = {path.stem: path for path in paths}
    if set(sample_ids) != set(images) or len(sample_ids) != len(images):
        raise ValueError("Kaggle sample submission and mounted test image IDs do not agree")
    with out.open("w", newline="") as handle, torch.inference_mode():
        writer = csv.writer(handle)
        writer.writerow(["image_id", "PredictionString"])
        for name in sample_ids:
            with Image.open(images[name]) as image:
                width, height = image.size
                tensor = pil_to_tensor(image.convert("RGB")).float().div_(255)
            result = model([tensor])[0]
            tokens = []
            for score, box in zip(result["scores"].tolist(), result["boxes"].tolist(), strict=True):
                if not all(math.isfinite(x) for x in (score, *box)) or not 0 <= score <= 1:
                    raise ValueError("Non-finite or invalid model output")
                x1, y1, x2, y2 = box
                left, top = max(0, math.floor(x1)), max(0, math.floor(y1))
                right, bottom = min(width, math.ceil(x2)), min(height, math.ceil(y2))
                if right > left and bottom > top:
                    tokens.append(f"{score:.8f} {left} {top} {right-left} {bottom-top}")
            writer.writerow([name, " ".join(tokens)])
    print(f"Validated and wrote {len(sample_ids)} predictions")


if __name__ == "__main__":
    if os.environ.get("WHEAT_SUBMIT_TEST_ROOT"):
        predict(Path(os.environ["WHEAT_SUBMIT_TEST_ROOT"]), Path(os.environ["WHEAT_SUBMIT_WEIGHTS"]),
                Path(os.environ["WHEAT_SUBMIT_OUTPUT"]))
    else:
        mounted = Path("/kaggle/input")
        roots = sorted(path.parent for path in mounted.rglob("sample_submission.csv")
                       if path.parent.name == "global-wheat-detection")
        candidates = sorted(mounted.rglob("weights.pt"))
        if len(roots) != 1 or len(candidates) != 1:
            raise ValueError(f"Expected one competition mount and one baseline checkpoint; found {len(roots)} and {len(candidates)}")
        predict(roots[0], candidates[0], Path("/kaggle/working/submission.csv"))
